# notebook — HarborDesk's open book

Priya joins HarborDesk and asks: **How many annual leave days do I get?**

The chat model can write a fluent answer. It has never read HarborDesk's files. HarborDesk is fictional, so the number 18 is not in the model's memory. This notebook gives the model an open book.

```text
question ──► search the policy chunks ──► paste the best chunks ──► model writes the answer
```

That is **RAG**: Retrieval-Augmented Generation. We build it with LangChain, one object at a time.

| Step | What we do |
|---|---|
| 1 | Ask with no files |
| 2 | Load each policy as a `Document` and cut it into chunks |
| 3 | Turn text into numbers (embeddings) |
| 4 | Save those numbers in Chroma and search |
| 5 | Answer only from the retrieved chunks |
| 6 | See two failures |
| 7 | Join search and answer into one chain |
| 8 | Draw that chain as a LangGraph |

**Before you run.** Select the kernel **RAG Env (Python 3.11)**. Ollama must be running, with `qwen3.5:4b` and `qwen3-embedding:4b`. Run all cells from the top. Each cell uses the objects created above it.


## Step 0 — The model that writes

**What.** One LangChain chat model, `ChatOllama`.
**Why.** Every later answer uses this same writer. Temperature 0 keeps the sentence stable, so you can compare runs.


In [1]:
from pathlib import Path

from langchain_chroma import Chroma                       # stores chunks and searches them
from langchain_core.documents import Document             # text plus labels (metadata)
from langchain_core.embeddings import Embeddings          # the interface embed_documents / embed_query
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = Path("corpus")                 # the seven HarborDesk policy files, next to this notebook
CHAT_MODEL = "qwen3.5:4b"               # writes answers
EMBED_MODEL = "qwen3-embedding:4b"      # turns text into numbers

# temperature 0: pick the most likely next word, so the answer does not wander.
llm = ChatOllama(model=CHAT_MODEL, temperature=0, seed=7, num_ctx=4096, reasoning=False, num_predict=180)

print("policy files:", len(list(CORPUS.glob("*.md"))))


policy files: 7


## Step 1 — Ask with no handbook

**What.** Send Priya's question to `llm.invoke`. No policy is attached.
**Why.** This is the assistant before RAG. A careful model says it does not know.
**You should see.** A refusal. Not "18 days".


In [2]:
# The question we will reuse in every later step.
question = "How many annual leave days does a full-time HarborDesk employee receive?"

# .invoke sends the string. .content is the reply text.
reply = llm.invoke(question + "\nIf you do not have HarborDesk's documents, say so.")
print(reply.content)


I do not have access to specific internal HR policies or employment contracts for **HarborDesk** (or any other private company). Employment terms such as annual leave days vary by country of operation and are typically detailed in the employee handbook or contract provided directly by the employer.

To get an accurate answer, you should:
1. Check your onboarding packet or employee handbook.
2. Log into HarborDesk's internal portal (if available) to view HR policies.
3. Contact their Human Resources department directly via email or phone.


The model did not name a HarborDesk file, because none was given to it. Next we load the handbook.


## Step 2 — One file becomes chunks

**What.** A LangChain `Document` is the policy text plus labels. A splitter cuts that text into chunks of about 500 characters.
**Why.** Search returns a chunk, not the whole handbook. The chunk still needs its document id, so the model can cite `[HR-LEAVE-2026]`.

**Example.** The first four lines of a file are labels:

```text
Doc id: HR-LEAVE-2026   →  metadata["doc_id"]  = "HR-LEAVE-2026"
Status: current         →  metadata["status"]  = "current"
Year: 2026              →  metadata["year"]    = "2026"
Title: Annual leave policy
```

The lines under the header are `page_content`. `HR-LEAVE-2023` is an old file: 15 days, status `archived`. The current file says 18.


In [3]:
def load_policy(path: Path) -> Document:
    """First 4 lines are labels. The rest is the policy text."""
    lines = path.read_text(encoding="utf-8").splitlines()
    header, body = lines[:4], lines[4:]
    meta = {"source": path.name}
    for line in header:                              # "Doc id: HR-LEAVE-2026"
        key, value = line.split(":", 1)
        meta[key.strip().lower().replace(" ", "_")] = value.strip()
    return Document(page_content="\n".join(body).strip(), metadata=meta)


def make_chunks(policies: list[Document], chunk_size: int = 500) -> list[Document]:
    """Cut each policy. Stamp the document id onto every piece so it can be cited."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=40,                            # keep a sentence that lands on the cut
        separators=["\n## ", "\n\n", "\n", " "],  # prefer a heading, then a paragraph, then a word
    )
    chunks = splitter.split_documents(policies)      # metadata is copied onto each chunk
    for chunk in chunks:
        m = chunk.metadata
        chunk.page_content = (
            f"Document {m['doc_id']} ({m['status']}, {m['year']}). {m['title']}.\n"
            + chunk.page_content
        )
    return chunks


policies = [load_policy(p) for p in sorted(CORPUS.glob("*.md"))]
chunks = make_chunks(policies)

print(f"{len(policies)} policies -> {len(chunks)} chunks\n")
for policy in policies:
    print(f"{policy.metadata['doc_id']:16} {policy.metadata['status']:9} {policy.metadata['title']}")


7 policies -> 9 chunks

FAC-FIRE-2026    current   Fire evacuation
FIN-EXP-2026     current   Travel expense policy
FIN-PRINT-2026   current   Office operations note
HR-LEAVE-2023    archived  Annual leave policy (archived)
HR-LEAVE-2026    current   Annual leave policy
IT-DEVICE-2026   current   Laptop and loss reporting
OPS-SEV-2026     current   Incident severity clocks


In [4]:
# Read the old rule. We look it up by id, not by position in the list.
archived = next(chunk for chunk in chunks if chunk.metadata["doc_id"] == "HR-LEAVE-2023")
print(archived.page_content)


Document HR-LEAVE-2023 (archived, 2023). Annual leave policy (archived).
## Superseded rule

This document is kept for audit. It was superseded by HR-LEAVE-2026. Do not use it for a current employee question.

## Old entitlement

Under the 2023 rule, a full-time HarborDesk employee received 15 working days of annual leave in each calendar year. Up to 8 unused days could be carried forward. That carry-forward rule is no longer in force.


That chunk says **15 days** and `archived`. The current rule, in `HR-LEAVE-2026`, is **18 days**. Both chunks mention annual leave, so a meaning search can rank them close together.


## Step 3 — Turn text into numbers

**What.** An **embedding** is a list of numbers that stands for the meaning of a text. Similar meaning means similar lists.
**Why.** Chroma cannot search "meaning" until each chunk is a list of numbers.

The **index** is that saved store: chunks, their numbers, and their labels. We build it once in the next step. After that, only the question is turned into numbers.

Qwen3-Embedding was trained with an instruction on the **question only**:

```text
question  →  Instruct: <task>\nQuery:<question>  →  2560 numbers
policy    →  the text as written                  →  2560 numbers
```

**Example.** Both sentences below contain "leave". One is annual leave. One is a fire drill. The scores should show which meaning the question asked for.


In [5]:
# This sentence is for questions only. Do not put it in front of a policy.
TASK = "Given a HarborDesk employee question, retrieve the policy passage that answers it"


class QwenEmbeddings(Embeddings):
    """Policies are embedded as written. Questions get the instruction prefix."""

    def __init__(self):
        self.model = OllamaEmbeddings(model=EMBED_MODEL)

    def embed_documents(self, texts):
        # Used while the index is built. No instruction, or the policy vector drifts.
        return self.model.embed_documents(texts)

    def embed_query(self, text):
        # Used when Priya asks. The prefix is the format this model was trained with.
        return self.model.embed_query(f"Instruct: {TASK}\nQuery:{text}")


def cosine(a, b):
    """1 = same direction. 0 = unrelated. Length is divided out, so a longer list is not automatically closer."""
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (sum(x * x for x in a) ** 0.5 * sum(y * y for y in b) ** 0.5)


embeddings = QwenEmbeddings()
query_vec = embeddings.embed_query(question)
leave_vec, fire_vec = embeddings.embed_documents([
    "A full-time employee receives 18 working days of annual leave.",
    "When the alarm sounds, leave the building by the nearest stairwell.",
])

print("numbers per text:", len(query_vec))
print("question vs annual leave:", round(cosine(query_vec, leave_vec), 3))
print("question vs fire drill:  ", round(cosine(query_vec, fire_vec), 3))


numbers per text: 2560
question vs annual leave: 0.695
question vs fire drill:   0.278


The annual-leave sentence should score higher (about 0.70). The fire sentence shares the word "leave" and should score lower (about 0.28).

A high score means the meanings are close. It does not prove the sentence is HarborDesk's current rule. The next step still has to find the right file.


## Step 4 — Save the index and search

**What.** `Chroma.from_documents` calls `embed_documents` on every chunk and saves the index in `chroma_db/`.
**Why.** A question then calls `embed_query`, and Chroma returns the nearest chunks.
**You should see.** `HR-LEAVE-2026` at rank 1, and the archived `HR-LEAVE-2023` very close behind it.


In [6]:
# embedding= is required: Chroma uses it to call embed_documents on the chunks.
store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f"student-{i}" for i in range(len(chunks))],  # fixed ids: re-running replaces rows
    collection_name="harbordesk_student",               # separate from the class notebook's collection
    persist_directory="chroma_db",
    collection_metadata={"hnsw:space": "cosine"},       # rank by cosine similarity
)


def search(question: str, k: int = 4) -> list[Document]:
    """Print the k nearest chunks. Higher score = closer meaning."""
    hits = store.similarity_search_with_relevance_scores(question, k=k)
    print("Q:", question)
    for rank, (doc, score) in enumerate(hits, start=1):
        preview = doc.page_content.splitlines()[1][:70]
        print(f"  {rank}. {doc.metadata['doc_id']:16} {score:.3f}  {preview}")
    return [doc for doc, _ in hits]


leave_hits = search(question)


Q: How many annual leave days does a full-time HarborDesk employee receive?
  1. HR-LEAVE-2026    0.879  ## Entitlement
  2. HR-LEAVE-2023    0.854  ## Superseded rule
  3. HR-LEAVE-2026    0.635  ## Joiners
  4. FIN-PRINT-2026   0.418  Reception issues visitor badges. Parking permits renew each April. The


## Try this

`leave_hits` is the list just printed. Before you read the answer cell, write one line that prints `True` if `HR-LEAVE-2023` is in that list.


In [7]:
# Your line:
print("archived policy in the top 4:", any(doc.metadata["doc_id"] == "HR-LEAVE-2023" for doc in leave_hits))


archived policy in the top 4: True


`True` means search found similar text, including the old 15-day rule. The embedding does not read `status: archived` as "do not use". That label is only metadata. Notebook 2 filters on it before the model sees the page.


## Step 5 — Answer from the chunks

**What.** Paste the retrieved chunks into a prompt, then call the same `ChatOllama`.
**Why.** The model writes. The index supplies the fact.

The strict rules are the contract: use only these passages, cite the document id, and say `NOT_IN_CORPUS` if the answer is missing.


In [8]:
STRICT = """You are a HarborDesk policy assistant.
Use only the passages. Cite the document id for every fact, like [HR-LEAVE-2026].
If a passage is archived, do not state it as the current rule.
If the passages do not contain the answer, reply exactly: NOT_IN_CORPUS"""


def answer(question: str, docs: list[Document]) -> str:
    """Passages + rules + question → one reply."""
    passages = "\n\n---\n\n".join(doc.page_content for doc in docs)
    prompt = f"{STRICT}\n\nPassages:\n{passages}\n\nQuestion: {question}"
    return llm.invoke(prompt).content


print(answer(question, leave_hits))


A full-time HarborDesk employee receives 18 working days of annual leave in each calendar year [HR-LEAVE-2026].


You should see **18 days** and `[HR-LEAVE-2026]`.

The archived chunk was still inside the prompt. The instruction asked the model to ignore it. That is a request, not a guarantee.


## Step 6 — Two questions the search does not fully solve

**Sabbatical.** No file states a length. Search still returns the nearest leave chunks. The strict prompt should say `NOT_IN_CORPUS`.

**`MEALDOM15`.** Only `FIN-PRINT-2026` contains that exact code. Embeddings match meaning, and a code has very little meaning, so rank 1 can be a file that does not contain the word.


In [9]:
for probe in [
    "How long is the HarborDesk sabbatical?",   # not in any file
    "MEALDOM15",                                  # exact code, in one office-notes sentence
]:
    hits = search(probe, k=3)
    print("A:", answer(probe, hits))
    if probe == "MEALDOM15":
        print("rank 1 contains MEALDOM15:", "MEALDOM15" in hits[0].page_content)
    print()


Q: How long is the HarborDesk sabbatical?
  1. HR-LEAVE-2026    0.752  ## Entitlement
  2. HR-LEAVE-2023    0.720  ## Superseded rule
  3. HR-LEAVE-2026    0.487  ## Joiners


A: NOT_IN_CORPUS



Q: MEALDOM15
  1. FIN-EXP-2026     0.550  ## Policy codes
  2. FIN-EXP-2026     0.549  ## Meal allowances
  3. FIN-PRINT-2026   0.546  Reception issues visitor badges. Parking permits renew each April. The


A: The document FIN-PRINT-2026 states that "the legacy reimbursement PDF prints the field MEALDOM15" and explicitly defines this as "[HR-LEAVE-2026] The field is the domestic meal allowance." However, since Document FIN-PRINT-2026 is an office operations note rather than a policy document defining allowances (unlike FIN-EXP-2026), we must verify if it contains specific rules or just operational notes.

The passage in FIN-PRINT-2026 says: "That field [MEALDOM15] is the domestic meal allowance." This directly answers what MEALDOM15 represents within that document's context, even though it describes a legacy PDF behavior rather than an active policy rule for current reimbursement calculations (which are defined in FIN-EXP-2026).

Based on
rank 1 contains MEALDOM15: False



Nearest chunk is not the same as "this chunk answers the question". Search always returns something. A code can lose to a paragraph about meals.


## Step 7 — The same steps as one chain

**What.** `|` passes the output of one LangChain step into the next.
**Why.** Step 4 searched, then Step 5 answered. This cell is those two calls written once.

```text
question ──► retriever ──► chunks ──► one string ──► prompt ──► ChatOllama ──► text
```

Pass the question **string** to `chain.invoke`. The chain builds `{context, question}` itself.


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# as_retriever wraps the Chroma store. k=4 is the same "top 4" as search().
retriever = store.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    """Join chunks the way answer() did, with --- between them."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template(STRICT + "\n\nPassages:\n{context}\n\nQuestion: {question}")

# Read the pipe left to right: find chunks, fill the prompt, write, return a string.
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke(question))
print()
print(chain.invoke("How long is the HarborDesk sabbatical?"))


A full-time HarborDesk employee receives 18 working days of annual leave in each calendar year [HR-LEAVE-2026].



NOT_IN_CORPUS


Same answers as Step 5 and Step 6: **18 days**, then `NOT_IN_CORPUS`. The chain did not make the model smarter. It wired search and the prompt together.

Every question still takes the same path: search, then write. A greeting, a missing policy, and a bad citation all go that way.


## Step 8 — That chain, drawn as a graph

**What.** LangGraph has three pieces.

| Piece | Meaning here |
|---|---|
| State | One dictionary: the question, the chunks, the answer, the path |
| Node | A function. `retrieve`, then `generate` |
| Edge | Which node runs next. These edges are fixed |

**Why.** A chain and this graph do the same work. The graph is the shape that can later choose: greet, retry, or refuse. This notebook does not add those choices yet.


In [11]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class StraightState(TypedDict):
    question: str
    hits: list
    answer: str
    steps: Annotated[list[str], operator.add]   # append each line, so the path grows


def retrieve_node(state: StraightState) -> dict:
    docs = retriever.invoke(state["question"])
    names = ", ".join(doc.metadata["doc_id"] for doc in docs)
    return {"hits": docs, "steps": [f"retrieve -> {names}"]}


def generate_node(state: StraightState) -> dict:
    text = (prompt | llm | StrOutputParser()).invoke({
        "context": format_docs(state["hits"]),
        "question": state["question"],
    })
    return {"answer": text, "steps": ["generate"]}


graph = StateGraph(StraightState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")       # every question searches
graph.add_edge("retrieve", "generate")  # search always continues to writing
graph.add_edge("generate", END)
assistant = graph.compile()
assistant.get_graph().print_ascii()

result = assistant.invoke({"question": question, "hits": [], "answer": "", "steps": []})
print("PATH:", " → ".join(result["steps"]))
print(result["answer"])


+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| retrieve |   
+----------+   
      *        
      *        
      *        
+----------+   
| generate |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


PATH: retrieve -> HR-LEAVE-2026, HR-LEAVE-2023, HR-LEAVE-2026, FIN-PRINT-2026 → generate
A full-time HarborDesk employee receives 18 working days of annual leave in each calendar year [HR-LEAVE-2026].


The path is `retrieve → generate`. There is no node that refuses. `NOT_IN_CORPUS` appears only when the prompt inside `generate` asks for it. The archived chunk can still be in `hits`.
